# Walmart Sales — Régression supervisée

**Projet :** Bloc 3 — Jedha CDSD
**Auteur :** Aymeric Lahonde

## Objectif

Walmart veut prédire les ventes hebdomadaires de ses magasins à partir de variables temporelles et économiques. On va construire :
- Part 1 : EDA + preprocessing
- Part 2 : Régression linéaire baseline
- Part 3 : Régularisation (Ridge / Lasso) pour éviter l'overfitting


---
## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error

sns.set_theme(style='whitegrid')
print('Imports OK')


---
## Part 1 — EDA + Preprocessing

On commence par charger le dataset et l'explorer.


In [ ]:
df = pd.read_csv('../data/walmart_sales.csv')
print('Shape :', df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


### 1.1 Drop des lignes sans target


In [ ]:
df = df.dropna(subset=['Weekly_Sales']).copy()
print(f'Apres drop : {df.shape}')


### 1.2 Feature engineering depuis Date


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df = df.drop(columns=['Date'])
df.head()


### 1.3 Drop des outliers (méthode 3-sigma)


In [ ]:
outlier_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
for col in outlier_cols:
    mu, sigma = df[col].mean(), df[col].std()
    df = df[(df[col] >= mu - 3*sigma) & (df[col] <= mu + 3*sigma)]
print(f'Apres drop outliers : {df.shape}')


### 1.4 Visualisation


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, outlier_cols):
    sns.histplot(df[col], ax=ax, bins=30)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
numeric = df.select_dtypes(include='number')
corr = numeric.corr()['Weekly_Sales'].sort_values(ascending=False)
print(corr)


### 1.5 Préparation X et y + train/test split


In [ ]:
y = df['Weekly_Sales']
X = df.drop(columns=['Weekly_Sales'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train :', X_train.shape, '| Test :', X_test.shape)


In [ ]:
categorical = ['Store', 'Holiday_Flag']
numerical = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Day', 'DayOfWeek']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
])


---
## Part 2 — Régression linéaire baseline


In [ ]:
pipe_lr = Pipeline([
    ('prep', preprocessor),
    ('model', LinearRegression()),
])

pipe_lr.fit(X_train, y_train)

r2_train = pipe_lr.score(X_train, y_train)
r2_test = pipe_lr.score(X_test, y_test)
rmse_test = np.sqrt(mean_squared_error(y_test, pipe_lr.predict(X_test)))

print(f'R2 train  : {r2_train:.3f}')
print(f'R2 test   : {r2_test:.3f}')
print(f'RMSE test : {rmse_test:.0f} $')


In [ ]:
feature_names = pipe_lr.named_steps['prep'].get_feature_names_out()
coefs = pipe_lr.named_steps['model'].coef_
df_coefs = pd.DataFrame({'feature': feature_names, 'coef': coefs}).sort_values('coef', key=abs, ascending=False)
df_coefs.head(15)


---
## Part 3 — Régularisation (Ridge / Lasso)


In [ ]:
pipe_ridge = Pipeline([('prep', preprocessor), ('model', Ridge(alpha=1.0))])
pipe_ridge.fit(X_train, y_train)
print(f'Ridge - R2 test : {pipe_ridge.score(X_test, y_test):.3f}')


In [ ]:
pipe_lasso = Pipeline([('prep', preprocessor), ('model', Lasso(alpha=1.0, max_iter=10000))])
pipe_lasso.fit(X_train, y_train)
print(f'Lasso - R2 test : {pipe_lasso.score(X_test, y_test):.3f}')


### 3.1 Tuning de alpha avec GridSearchCV


In [ ]:
param_grid = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}

grid_ridge = GridSearchCV(
    Pipeline([('prep', preprocessor), ('model', Ridge())]),
    param_grid, cv=5, scoring='r2', n_jobs=-1,
)
grid_ridge.fit(X_train, y_train)

print(f'Best alpha (Ridge) : {grid_ridge.best_params_}')
print(f'Best R2 CV         : {grid_ridge.best_score_:.3f}')
print(f'R2 test            : {grid_ridge.score(X_test, y_test):.3f}')


---
## Bilan

| Modele | R2 train | R2 test | RMSE test |
|---|---|---|---|
| LinearRegression | TODO | TODO | TODO |
| Ridge (alpha=1) | - | TODO | - |
| Lasso (alpha=1) | - | TODO | - |
| Ridge tune (GridSearchCV) | - | TODO | - |

**Conclusions :**
- Quelle est la feature la plus importante ?
- Y a-t-il de l'overfitting ? Si oui, est-ce que la regularisation l'a reduit ?
- Pistes d'amelioration : autre modele, plus de features, polynomial, etc.
